# SCRFD Competition Offline Runner

Notebook template for Kaggle competition environments with internet disabled.

Expected Kaggle Dataset inputs:
- `scrfd-fullsearch-kaggle-src`: contains `scrfd-fullsearch-kaggle-offline.zip`
- `scrfd-wheelhouse`: optional wheel bundle if the competition image is missing packages
- `widerface-retinaface-format`: your WIDER FACE dataset in SCRFD format

In [ ]:
REPO_ZIP = ""
DATA_ROOT = ""
WHEELHOUSE = "/kaggle/input/scrfd-wheelhouse"

RUN_MODE = "baseline_train_quick"
# Options: baseline_train_quick, baseline_train, baseline_eval, step1_generate, step1_train, step1_eval,
# step1_stat, step2_generate, step2_train, step2_eval, step2_stat, full_search

GPU_ID = 0
TOTAL_EPOCHS = 16
EVAL_INTERVAL = 4
CHECKPOINT_INTERVAL = 4
IDX_FROM = 1
IDX_TO = 64
TEMPLATE = 10
CHECKPOINT = ""
THR = "0.02"
MODE_VALUE = "0"


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

def find_repo_source(explicit_path: str) -> Path:
    if explicit_path:
        candidate = Path(explicit_path)
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    zip_names = ['scrfd-fullsearch-kaggle-offline.zip', 'scrfd-fullsearch-kaggle-kagglefix-v2.zip']
    for zip_name in zip_names:
        matches = list(input_root.rglob(zip_name))
        if matches:
            return matches[0]
    for candidate in input_root.rglob('scrfd-fullsearch-kaggle'):
        if (candidate / 'scripts' / 'kaggle_competition_entry.py').exists():
            return candidate
    raise FileNotFoundError('Could not find SCRFD source dataset under /kaggle/input')

repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
if repo_root.exists():
    shutil.rmtree(repo_root)

repo_source = find_repo_source(REPO_ZIP)
print('Using repo source:', repo_source)
if repo_source.is_dir():
    source_root = repo_source / 'scrfd-fullsearch-kaggle'
    if source_root.exists():
        shutil.copytree(source_root, repo_root)
    else:
        shutil.copytree(repo_source, repo_root)
else:
    with zipfile.ZipFile(repo_source) as zf:
        zf.extractall('/kaggle/working')

assert repo_root.exists(), f'Missing repo after extraction: {repo_root}'
for shell_path in (repo_root / 'scripts').glob('*.sh'):
    shell_path.chmod(0o755)
for shell_path in (repo_root / 'tools').glob('*.sh'):
    shell_path.chmod(0o755)
for shell_path in (repo_root / 'search_tools').glob('*.sh'):
    shell_path.chmod(0o755)
os.chdir(repo_root)
print('Repo ready at', repo_root)


In [ ]:
import subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path\n\ndef _is_retinaface_root(candidate: Path) -> bool:\n    return (candidate / 'train' / 'labelv2.txt').exists() and (candidate / 'val' / 'labelv2.txt').exists()\n\ndef materialize_data_root(explicit_path: str) -> Path:\n    input_root = Path('/kaggle/input')\n    work_data_root = Path('/kaggle/working/retinaface_dataset')\n\n    def from_candidate(candidate: Path) -> Path | None:\n        if not candidate.exists():\n            return None\n        if candidate.is_dir():\n            if _is_retinaface_root(candidate):\n                return candidate\n            nested = candidate / 'retinaface'\n            if _is_retinaface_root(nested):\n                return nested\n            return None\n        if candidate.suffix.lower() == '.zip':\n            if work_data_root.exists():\n                shutil.rmtree(work_data_root)\n            work_data_root.mkdir(parents=True, exist_ok=True)\n            with zipfile.ZipFile(candidate) as zf:\n                zf.extractall(work_data_root)\n            extracted = work_data_root / 'retinaface'\n            if _is_retinaface_root(extracted):\n                return extracted\n        return None\n\n    if explicit_path:\n        resolved = from_candidate(Path(explicit_path))\n        if resolved is not None:\n            return resolved\n\n    for candidate in input_root.rglob('retinaface'):\n        resolved = from_candidate(candidate)\n        if resolved is not None:\n            return resolved\n    for candidate in input_root.rglob('retinaface-kaggle-upload-with-test.zip'):\n        resolved = from_candidate(candidate)\n        if resolved is not None:\n            return resolved\n    for candidate in input_root.rglob('*.zip'):\n        if 'retinaface' in candidate.name.lower() or 'widerface' in candidate.name.lower():\n            resolved = from_candidate(candidate)\n            if resolved is not None:\n                return resolved\n    raise FileNotFoundError('Could not find retinaface dataset root or retinaface dataset zip under /kaggle/input')\n\nrepo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')\nentry = repo_root / 'scripts' / 'kaggle_competition_entry.py'\nwork_root = repo_root / 'work_dirs'\nresult_root = repo_root / 'wouts'\ndata_root = materialize_data_root(DATA_ROOT)\nprint('Using data root:', data_root)\n\ndef run_baseline_train_quick_direct() -> None:\n    work_dir = work_root / 'scrfd_1g_quick'\n    cfg_options = [\n        f'data.train.ann_file={data_root}/train/labelv2.txt',\n        f'data.train.img_prefix={data_root}/train/images/',\n        f'data.val.ann_file={data_root}/val/labelv2.txt',\n        f'data.val.img_prefix={data_root}/val/images/',\n        f'data.test.ann_file={data_root}/val/labelv2.txt',\n        f'data.test.img_prefix={data_root}/val/images/',\n        f'total_epochs={TOTAL_EPOCHS}',\n        f'evaluation.interval={EVAL_INTERVAL}',\n        f'checkpoint_config.interval={CHECKPOINT_INTERVAL}',\n    ]\n    cmd = [\n        sys.executable,\n        str(repo_root / 'tools' / 'train.py'),\n        str(repo_root / 'configs' / 'scrfd' / 'scrfd_1g.py'),\n        '--gpu-ids', str(GPU_ID),\n        '--work-dir', str(work_dir),\n        '--cfg-options', *cfg_options,\n    ]\n    print('Running direct quick baseline command:')\n    print(' '.join(cmd))\n    subprocess.run(cmd, check=True)\n\nif RUN_MODE == 'baseline_train_quick':\n    run_baseline_train_quick_direct()\nelse:\n    cmd = [\n        sys.executable,\n        str(entry),\n        '--mode', RUN_MODE,\n        '--data-root', str(data_root),\n        '--work-root', str(work_root),\n        '--result-root', str(result_root),\n        '--gpu-id', str(GPU_ID),\n        '--idx-from', str(IDX_FROM),\n        '--idx-to', str(IDX_TO),\n        '--template', str(TEMPLATE),\n        '--thr', str(THR),\n        '--mode-value', str(MODE_VALUE),\n    ]\n\n    if TOTAL_EPOCHS and RUN_MODE == 'baseline_train':\n        print('Note: current source bundle may ignore TOTAL_EPOCHS unless it is the refreshed version.')\n    if CHECKPOINT:\n        cmd.extend(['--checkpoint', CHECKPOINT])\n\n    if Path(WHEELHOUSE).exists():\n        cmd.extend(['--wheelhouse', WHEELHOUSE])\n    else:\n        cmd.append('--skip-offline-install')\n\n    print('Running command:')\n    print(' '.join(cmd))\n    subprocess.run(cmd, check=True)\n